In [ ]:
import os
import torch
import subprocess

# 1. Clone the repository if not present
if not os.path.exists('main.py'):
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL

# 2. Setup Dependencies
# FORCE REINSTALL TORCH to a stable version compatible with PyG wheels
# Kaggle's default torch 2.9.0+ is too new and causes slow source compilation for scatter/sparse
print("Downgrading PyTorch to stable 2.4.0 to ensure wheel compatibility...")
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# Now we can safely install wheels
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cu121.html

requirements_content = """colorlog
tensorboard
six
colorama
pyyaml
pandas
hyperopt==0.2.5
numpy<2.0.0
torch_geometric
# python>=3.9.18
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("Installing other requirements...")
!pip install -r requirements.txt

# 3. Dataset Preprocessing
if not os.path.exists('dataset/QB-video'):
    os.makedirs('dataset/QB-video')

if os.path.exists('dataset/QB-video.csv') and not os.path.exists('dataset/QB-video/QB-video.inter'):
    os.rename('dataset/QB-video.csv', 'dataset/QB-video/QB-video.inter')

# 4. Configuration Adjustment
config_path = 'properties/QB-video.yaml'
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        content = f.read()
    if 'field_separator' not in content:
        new_content = content.replace('inter: [user_id, item_id]', 'inter: [user_id, item_id]\n\nfield_separator: ","')
        with open(config_path, 'w') as f:
            f.write(new_content)

# 5. Run Training
!python main.py --dataset QB-video